[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/danpele/Time-Series-Analysis/blob/main/EN/Course_Notebooks/chapter12_lecture_notebook.ipynb)

---

# Chapter 12: Spectral Analysis and Frequency-Domain Methods

**Course:** Time Series Analysis and Forecasting  
**Program:** Bachelor program, Faculty of Cybernetics, Statistics and Economic Informatics, Bucharest University of Economic Studies, Romania  
**Academic Year:** 2025-2026

---

## Learning Objectives

By the end of this chapter, you will be able to:

1. Understand Fourier analysis and the Discrete Fourier Transform (DFT)
2. Compute and interpret the periodogram as a spectral density estimator
3. Apply advanced spectral estimation methods: Welch, multitaper, Blackman-Tukey
4. Perform cross-spectral analysis (coherence, phase, gain)
5. Extract business cycles using band-pass filters (HP, Baxter-King, Christiano-Fitzgerald)
6. Apply wavelet analysis to capture time-varying frequency content
7. Identify cycles in real economic data (GDP, sunspots)

## Setup and Imports

In [ ]:
# Install required packages (for Colab)
import sys
import os
if 'google.colab' in sys.modules:
    !pip install pywavelets statsmodels scipy numpy pandas matplotlib -q

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from scipy import signal
from scipy.fft import fft, fftfreq, ifft
import statsmodels.api as sm
from statsmodels.tsa.filters.hp_filter import hpfilter
from statsmodels.tsa.filters.bk_filter import bkfilter
from statsmodels.tsa.filters.cf_filter import cffilter

try:
    import pywt
    HAS_PYWT = True
except ImportError:
    HAS_PYWT = False
    print('pywt not available. Install with: pip install PyWavelets')

# ── Course color scheme ──────────────────────────────────────
COLORS = {
    'blue':       '#1A3A6E',
    'red':        '#DC3545',
    'green':      '#2E7D32',
    'amber':      '#B5853F',
    'orange':     '#E6802E',
    'purple':     '#8E44AD',
    'gray':       '#666666',
    'light_blue': '#5B8BD4',
}

# ── Global plot style ────────────────────────────────────────
plt.rcParams.update({
    'figure.figsize':    (12, 5),
    'font.size':         11,
    'axes.facecolor':    'none',
    'figure.facecolor':  'none',
    'axes.grid':         False,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'legend.frameon':    False,
})

print('Setup complete!')

---
## Section 1: Fourier Analysis and the Discrete Fourier Transform

Any stationary time series can be decomposed into a sum of sinusoidal components:

$$x(t) = \sum_{j} A_j \cos(2\pi f_j t + \phi_j)$$

where $f_j$ are the **frequencies**, $A_j$ are the **amplitudes**, and $\phi_j$ are the **phases**.

The **Discrete Fourier Transform (DFT)** of a length-$T$ sequence $\{x_t\}$ is:

$$d(\omega_j) = \frac{1}{\sqrt{T}} \sum_{t=1}^{T} x_t e^{-2\pi i \omega_j t}, \quad \omega_j = \frac{j}{T}, \; j = 0, 1, \ldots, T-1$$

The **Nyquist frequency** $f_N = 1/2$ (in cycles per sample) is the highest detectable frequency.

**Aliasing** occurs when a signal contains frequencies above the Nyquist limit — they fold back and appear as lower frequencies.

In [ ]:
# ── 1.1 Synthesize a multi-frequency signal ──────────────────
np.random.seed(42)

T  = 256          # number of observations
dt = 1.0          # sampling interval (1 year)
t  = np.arange(T) * dt

# True frequencies and amplitudes
f1, A1 = 0.10, 3.0    # 10-year cycle  (e.g. sunspot-like)
f2, A2 = 0.25, 1.5    # 4-year cycle   (e.g. political cycle)
f3, A3 = 0.05, 2.0    # 20-year cycle  (e.g. Kuznets swing)

noise = np.random.normal(0, 0.8, T)
x = (A1 * np.cos(2 * np.pi * f1 * t) +
     A2 * np.cos(2 * np.pi * f2 * t) +
     A3 * np.cos(2 * np.pi * f3 * t) +
     noise)

# ── 1.2 Compute DFT and amplitude spectrum ───────────────────
X     = fft(x)
freqs = fftfreq(T, d=dt)         # cycles per year
pos   = freqs > 0                 # keep only positive frequencies
amps  = 2 * np.abs(X[pos]) / T   # two-sided -> one-sided amplitude

# ── 1.3 Plot: signal + amplitude spectrum ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: time series
axes[0].plot(t, x, color=COLORS['blue'], lw=1.2, label='Synthetic signal')
axes[0].set_xlabel('Time (years)')
axes[0].set_ylabel('Amplitude')
axes[0].set_title('Synthetic Multi-Frequency Signal')

# Right: amplitude spectrum
axes[1].stem(freqs[pos], amps, linefmt=COLORS['blue'],
             markerfmt='o', basefmt='k-')
for f, A, label in [(f1, A1, f'f={f1} (T=10yr)'),
                    (f2, A2, f'f={f2} (T=4yr)'),
                    (f3, A3, f'f={f3} (T=20yr)')]:
    axes[1].axvline(f, color=COLORS['red'], lw=1, ls='--', alpha=0.7)
    axes[1].annotate(label, (f, A + 0.1), fontsize=9,
                     color=COLORS['red'], ha='center')
axes[1].set_xlabel('Frequency (cycles/year)')
axes[1].set_ylabel('Amplitude')
axes[1].set_title('DFT Amplitude Spectrum')
axes[1].set_xlim(0, 0.5)

plt.tight_layout()
plt.show()
print('True amplitudes: A1=3.0, A2=1.5, A3=2.0')
print('Recovered peaks: see vertical dashed lines')

In [ ]:
# ── 1.4 Aliasing demonstration ───────────────────────────────
# A 0.9 cycles/sample signal sampled at fs=1 aliases to 0.1 cycles/sample

f_true   = 0.9           # true frequency (above Nyquist = 0.5)
f_alias  = 1.0 - f_true  # = 0.1  (apparent frequency after aliasing)
t_fine   = np.linspace(0, 10, 1000)  # continuous reference
t_samp   = np.arange(0, 10, 1)       # discrete samples (fs = 1)

x_true  = np.sin(2 * np.pi * f_true  * t_fine)
x_alias = np.sin(2 * np.pi * f_alias * t_fine)
x_samp  = np.sin(2 * np.pi * f_true  * t_samp)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t_fine, x_true,  color=COLORS['gray'],  lw=1.0, ls='--',
        label=f'True signal  (f={f_true})')
ax.plot(t_fine, x_alias, color=COLORS['blue'],  lw=1.5,
        label=f'Aliased signal (f={f_alias})')
ax.scatter(t_samp, x_samp, color=COLORS['red'], zorder=5, s=60,
           label='Sampled points')
ax.set_xlabel('Time')
ax.set_ylabel('Amplitude')
ax.set_title(f'Aliasing: True f={f_true} aliases to f={f_alias} (Nyquist = 0.5)')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=3)
plt.tight_layout()
plt.show()
print(f'Nyquist frequency = 0.5 cycles/sample')
print(f'True frequency {f_true} > 0.5  =>  aliases to {f_alias}')

---
## Section 2: The Periodogram — Sunspot Cycle

The **periodogram** is the squared modulus of the DFT, scaled to estimate the spectral density:

$$I(\omega_j) = \frac{1}{T} \left| \sum_{t=1}^{T} x_t e^{-2\pi i \omega_j t} \right|^2$$

It measures **how much of the total variance** of a series is attributable to oscillations at each frequency $\omega_j$.

**Important limitation:** The periodogram is an *inconsistent* estimator — its variance does **not** decrease as $T \to \infty$.

### Sunspot Data
Annual sunspot counts since 1700 are the classic example of a quasi-periodic series with an ~11-year solar cycle.

In [ ]:
# ── 2.1 Load sunspot data ─────────────────────────────────────
sunspots_data = sm.datasets.sunspots.load_pandas()
sunspots      = sunspots_data.data
sunspots.set_index('YEAR', inplace=True)
sunspots.index = sunspots.index.astype(int)

y = sunspots['SUNACTIVITY'].values
T = len(y)
print(f'Sunspot data: {T} annual observations ({sunspots.index[0]}-{sunspots.index[-1]})')
print(f'Mean: {y.mean():.1f}, Std: {y.std():.1f}')

# ── 2.2 Compute periodogram ───────────────────────────────────
freqs_p, power = signal.periodogram(y, fs=1.0)  # fs=1 -> freq in cycles/year

# Find dominant peak
peak_idx  = np.argmax(power[1:]) + 1   # skip DC component
peak_freq = freqs_p[peak_idx]
peak_per  = 1.0 / peak_freq
print(f'\nDominant cycle: {peak_per:.1f} years (frequency = {peak_freq:.4f} cycles/year)')

# ── 2.3 Plot: time series + periodogram ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: sunspot time series
axes[0].fill_between(sunspots.index, y, color=COLORS['amber'], alpha=0.4)
axes[0].plot(sunspots.index, y, color=COLORS['amber'], lw=1.0)
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Sunspot Number')
axes[0].set_title('Annual Sunspot Activity')

# Right: periodogram
axes[1].semilogy(freqs_p[1:], power[1:], color=COLORS['blue'], lw=1.2)
axes[1].axvline(peak_freq, color=COLORS['red'], lw=1.5, ls='--',
                label=f'~{peak_per:.1f}-year cycle (f={peak_freq:.3f})')
axes[1].set_xlabel('Frequency (cycles/year)')
axes[1].set_ylabel('Spectral Density (log scale)')
axes[1].set_title('Periodogram of Sunspot Activity')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)

plt.tight_layout()
plt.show()

---
## Section 3: Spectral Density Estimation

Because the raw periodogram is inconsistent (high variance), we use **smoothed estimators**:

| Method | Idea | Bias-Variance Trade-off |
|--------|------|--------------------------|
| **Raw periodogram** | Direct DFT squared | Low bias, high variance |
| **Welch's method** | Average over overlapping segments | Higher bias, lower variance |
| **Blackman-Tukey** | Smooth autocovariance, then FT | Controlled via lag window |
| **Multitaper (DPSS)** | Average over orthogonal tapers | Best concentration, low leakage |

**Spectral leakage** arises from the finite sample and creates side-lobes. Tapering (windowing) reduces leakage at the cost of reduced frequency resolution.

In [ ]:
# ── 3.1 Compare estimation methods on sunspot data ─────────

# (a) Raw periodogram
freqs_raw, psd_raw = signal.periodogram(y, fs=1.0)

# (b) Welch's method — 64-point segments, 50% overlap, Hann window
freqs_welch, psd_welch = signal.welch(y, fs=1.0, nperseg=64,
                                       noverlap=32, window='hann')

# (c) Multitaper via DPSS (discrete prolate spheroidal sequences)
NW = 4     # time-half-bandwidth product (controls resolution)
K  = 7     # number of tapers  (K = 2*NW - 1 is standard)
tapers, ratios = signal.windows.dpss(len(y), NW, Kmax=K, return_ratios=True)

psd_mt_all = np.zeros((K, len(freqs_raw)))
for k in range(K):
    f_k, p_k = signal.periodogram(y * tapers[k], fs=1.0)
    psd_mt_all[k] = p_k

# Weighted average (eigenvalue weights = concentration ratios)
weights  = ratios[:, np.newaxis]
psd_mt   = np.sum(weights * psd_mt_all, axis=0) / np.sum(weights)
freqs_mt = f_k

print(f'DPSS tapers: NW={NW}, K={K}')
print(f'Concentration ratios: {ratios.round(4)}')

# ── 3.2 Plot comparison ────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 6))

ax.semilogy(freqs_raw[1:],   psd_raw[1:],   color=COLORS['gray'],
            lw=0.8, alpha=0.6, label='Raw periodogram (noisy)')
ax.semilogy(freqs_welch[1:], psd_welch[1:], color=COLORS['orange'],
            lw=2.0, label="Welch's method (nperseg=64)")
ax.semilogy(freqs_mt[1:],    psd_mt[1:],    color=COLORS['blue'],
            lw=2.0, label=f'Multitaper DPSS (NW={NW}, K={K})')
ax.axvline(1/11, color=COLORS['red'], lw=1.5, ls='--', label='~11-year cycle')

ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Power Spectral Density (log scale)')
ax.set_title('Spectral Estimation Comparison — Sunspot Data')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.3 Blackman-Tukey estimator (lag-window method) ────────
# Compute sample autocovariance, apply Bartlett window, take FFT

T_sun      = len(y)
y_centered = y - y.mean()
max_lag    = 40    # truncation lag M

# Sample autocovariance
acov = np.array([np.mean(y_centered[:T_sun-h] * y_centered[h:])
                 for h in range(max_lag + 1)])

# Bartlett (triangular) window
lags     = np.arange(max_lag + 1)
bartlett = 1.0 - lags / (max_lag + 1)

# Windowed autocovariance -> two-sided sequence
windowed_acov = np.concatenate([bartlett * acov,
                                 (bartlett * acov)[-2:0:-1]])

# FFT of windowed autocovariance
psd_bt   = np.real(np.fft.rfft(windowed_acov))
freqs_bt = np.fft.rfftfreq(len(windowed_acov))

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
ax.semilogy(freqs_raw[1:], psd_raw[1:],        color=COLORS['gray'],
            lw=0.8, alpha=0.5, label='Raw periodogram')
ax.semilogy(freqs_bt[1:],  np.abs(psd_bt[1:]), color=COLORS['green'],
            lw=2.0, label=f'Blackman-Tukey (Bartlett, M={max_lag})')
ax.axvline(1/11, color=COLORS['red'], lw=1.5, ls='--', label='~11-year cycle')
ax.set_xlabel('Frequency (cycles/year)')
ax.set_ylabel('Power Spectral Density (log scale)')
ax.set_title('Blackman-Tukey Spectral Estimate — Sunspot Data')
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=3)
plt.tight_layout()
plt.show()
print(f'Truncation lag M={max_lag}: frequency resolution ~{1/(2*max_lag):.3f} cycles/year')

---
## Section 4: Cross-Spectral Analysis

Given two series $x_t$ and $y_t$, the **cross-spectrum** is:

$$f_{xy}(\omega) = \sum_{h=-\infty}^{\infty} \gamma_{xy}(h)\, e^{-2\pi i \omega h}$$

Key derived quantities:

- **Squared coherence:** $C^2_{xy}(\omega) = \dfrac{|f_{xy}(\omega)|^2}{f_x(\omega)\, f_y(\omega)} \in [0,1]$ — linear association at each frequency

- **Phase spectrum:** $\phi_{xy}(\omega) = \arg[f_{xy}(\omega)]$ — lead/lag relationship

- **Gain:** $G_{xy}(\omega) = |f_{xy}(\omega)| / f_x(\omega)$ — amplitude scaling from $x$ to $y$

In [ ]:
# ── 4.1 Generate two lead-lag related signals ────────────────
np.random.seed(2024)
N    = 512
t_cs = np.arange(N)

# x: noisy business-cycle signal (dominant 20-quarter period)
x_cs = (2.0 * np.sin(2 * np.pi * t_cs / 20) +
        1.0 * np.sin(2 * np.pi * t_cs / 8) +
        np.random.normal(0, 0.5, N))

# y: lags x by 5 periods at the 20-quarter cycle
lead = 5
y_cs = (1.5 * np.sin(2 * np.pi * (t_cs - lead) / 20) +
        0.8 * np.sin(2 * np.pi * t_cs / 8) +
        np.random.normal(0, 0.5, N))

# ── 4.2 Compute coherence and phase ──────────────────────────
freqs_coh, Cxy = signal.coherence(x_cs, y_cs, fs=4.0,
                                   nperseg=128, noverlap=64)
freqs_csd, Pxy = signal.csd(x_cs, y_cs, fs=4.0,
                              nperseg=128, noverlap=64)
phase = np.angle(Pxy, deg=True)

# ── 4.3 Plot coherence and phase ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Coherence
axes[0].plot(freqs_coh, Cxy, color=COLORS['blue'], lw=2.0)
axes[0].axhline(0.5, color=COLORS['gray'], ls=':', lw=1.0)
axes[0].axvline(4/20, color=COLORS['red'],   ls='--', lw=1.5,
                label='f=4/20 (20-quarter cycle)')
axes[0].axvline(4/8,  color=COLORS['orange'],ls='--', lw=1.5,
                label='f=4/8  (8-quarter cycle)')
axes[0].set_xlabel('Frequency (cycles/quarter)')
axes[0].set_ylabel('Squared Coherence')
axes[0].set_title('Squared Coherence: x vs y')
axes[0].set_ylim(0, 1.05)
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)

# Phase spectrum
axes[1].plot(freqs_csd, phase, color=COLORS['purple'], lw=1.5)
axes[1].axvline(4/20, color=COLORS['red'], ls='--', lw=1.5,
                label='f=4/20 (20-quarter cycle)')
axes[1].axhline(0, color=COLORS['gray'], ls=':', lw=1.0)
axes[1].set_xlabel('Frequency (cycles/quarter)')
axes[1].set_ylabel('Phase (degrees)')
axes[1].set_title('Phase Spectrum: x leads y')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=1)

plt.tight_layout()
plt.show()
print(f'At f=4/20: coherence = {np.interp(4/20, freqs_coh, Cxy):.3f}')
print(f'Expected phase at 20-quarter cycle: {360 * lead / 20:.1f} degrees')

---
## Section 5: Band-Pass Filters and Business Cycle Extraction

A **band-pass filter** isolates components whose period falls within a specified range $[p_L, p_H]$ quarters.

| Filter | Ideal? | Key parameter | Typical use |
|--------|--------|---------------|-------------|
| **HP (Hodrick-Prescott)** | No | $\lambda$ | Trend removal |
| **Baxter-King (BK)** | Approx. | K (leads/lags) | Business cycles |
| **Christiano-Fitzgerald (CF)** | Approx. | Random walk assumption | Business cycles |

For **quarterly data**, the NBER business cycle range is **6-32 quarters** (1.5-8 years).  
Standard HP filter parameter: $\lambda = 1600$ for quarterly data.

In [ ]:
# ── 5.1 Simulate quarterly GDP (log-level) ───────────────────
np.random.seed(100)
Q        = 200   # 50 years of quarterly data
quarters = pd.date_range('1970Q1', periods=Q, freq='QS')

# Trend: slow upward drift
trend_gdp = 7.0 + 0.006 * np.arange(Q) + 0.000002 * np.arange(Q)**2

# Business cycle: dominant 20-quarter period
bc_gdp = (0.015 * np.sin(2 * np.pi * np.arange(Q) / 20) +
           0.008 * np.sin(2 * np.pi * np.arange(Q) / 12))

# Irregular component (random walk drift)
irr_gdp  = np.cumsum(np.random.normal(0, 0.003, Q))
log_gdp  = pd.Series(trend_gdp + bc_gdp + irr_gdp,
                     index=quarters, name='log_GDP')

print(f'Simulated quarterly log-GDP: {Q} observations')
print(f'Mean: {log_gdp.mean():.3f}, Std: {log_gdp.std():.3f}')

# ── 5.2 Apply HP filter ──────────────────────────────────────
hp_cycle, hp_trend = hpfilter(log_gdp, lamb=1600)

# ── 5.3 Apply Baxter-King filter ─────────────────────────────
bk_cycle = bkfilter(log_gdp, low=6, high=32, K=12)

# ── 5.4 Apply Christiano-Fitzgerald filter ───────────────────
cf_cycle, cf_trend = cffilter(log_gdp, low=6, high=32, drift=True)

# ── 5.5 Plot ─────────────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(14, 12), sharex=True)

# Panel 1: Original + HP trend
axes[0].plot(log_gdp.index, log_gdp.values, color=COLORS['blue'],
             lw=1.5, label='Log GDP')
axes[0].plot(hp_trend.index, hp_trend.values, color=COLORS['red'],
             lw=2.0, ls='--', label='HP trend (lambda=1600)')
axes[0].set_ylabel('Log GDP')
axes[0].set_title('Log GDP with HP Trend')
axes[0].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

# Panel 2: HP cycle and CF cycle
axes[1].plot(hp_cycle.index, hp_cycle.values, color=COLORS['blue'],
             lw=1.5, label='HP cycle')
axes[1].plot(cf_cycle.index, cf_cycle.values, color=COLORS['orange'],
             lw=1.5, label='CF cycle (6-32 quarters)')
axes[1].axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
axes[1].set_ylabel('Cycle component')
axes[1].set_title('HP and CF Business Cycle Components')
axes[1].legend(loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=2)

# Panel 3: BK cycle
axes[2].plot(bk_cycle.index, bk_cycle.values, color=COLORS['green'],
             lw=1.5, label='BK cycle (6-32 quarters, K=12)')
axes[2].axhline(0, color=COLORS['gray'], lw=0.8, ls=':')
axes[2].set_ylabel('Cycle component')
axes[2].set_xlabel('Date')
axes[2].set_title('Baxter-King Band-Pass Filter (6-32 quarters)')
axes[2].legend(loc='upper center', bbox_to_anchor=(0.5, -0.1), ncol=1)

plt.tight_layout()
plt.show()

In [ ]:
# ── 5.6 Transfer functions of band-pass filters ──────────────
freqs_tf = np.linspace(0, 0.5, 500)

# Ideal band-pass: passes 1/32 to 1/6 cycles/quarter
f_low  = 1 / 32
f_high = 1 / 6
ideal_gain = ((freqs_tf >= f_low) & (freqs_tf <= f_high)).astype(float)

# HP filter approximate transfer function
lam   = 1600
omega = 2 * np.pi * freqs_tf
# HP high-pass gain: g(f) = 4*lambda*(1 - cos(2*pi*f))^2 / (1 + 4*lambda*(1-cos(2*pi*f))^2)
cos_term = (1 - np.cos(2 * np.pi * freqs_tf))**2
hp_gain  = 4 * lam * cos_term / (1 + 4 * lam * cos_term + 1e-30)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(freqs_tf, ideal_gain, color=COLORS['gray'],   lw=2, ls='--',
        label='Ideal band-pass (6-32 quarters)')
ax.plot(freqs_tf, hp_gain,   color=COLORS['blue'],   lw=2.0,
        label='HP filter gain (lambda=1600)')
ax.axvline(f_low,  color=COLORS['red'],    ls=':', lw=1.2,
           label=f'f=1/32 (32 quarters)')
ax.axvline(f_high, color=COLORS['orange'], ls=':', lw=1.2,
           label=f'f=1/6  (6 quarters)')
ax.set_xlabel('Frequency (cycles/quarter)')
ax.set_ylabel('Gain')
ax.set_title('Transfer Functions: Ideal Band-Pass vs HP Filter')
ax.set_xlim(0, 0.5)
ax.set_ylim(-0.05, 1.15)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.12), ncol=2)
plt.tight_layout()
plt.show()
print('Note: HP filter gain rises gradually — it passes some low-frequency noise')
print('BK and CF filters are closer to the ideal rectangular band-pass')

---
## Section 6: Wavelet Analysis

**Wavelets** overcome Fourier analysis's limitation of global frequency representation.  
The **Continuous Wavelet Transform (CWT)** is:

$$W(a, b) = \frac{1}{\sqrt{a}} \int_{-\infty}^{\infty} x(t) \, \psi^*\!\left(\frac{t - b}{a}\right) dt$$

where $\psi$ is the **mother wavelet**, $a > 0$ is the **scale** (inversely related to frequency), and $b$ is the **time location**.

The **Morlet wavelet** $\psi(t) = \pi^{-1/4} e^{i\omega_0 t} e^{-t^2/2}$ provides a good balance between time and frequency resolution.

The **scalogram** plots $|W(a,b)|^2$ — the wavelet power — as a function of time and period, showing how frequency content evolves over time.

In [ ]:
# ── 6.1 Wavelet analysis of sunspot data ─────────────────────
if HAS_PYWT:
    y_sun  = sunspots['SUNACTIVITY'].values.astype(float)
    years  = sunspots.index.values
    y_norm = (y_sun - y_sun.mean()) / y_sun.std()

    # Complex Morlet wavelet: bandwidth=1.5, center frequency=1.0
    wavelet   = 'cmor1.5-1.0'
    freqs_wt  = np.geomspace(1/40, 1/2, 60)  # cycles per year
    scales    = pywt.frequency2scale(wavelet, freqs_wt)

    # CWT
    coeffs, freqs_cwt = pywt.cwt(y_norm, scales, wavelet, sampling_period=1.0)
    power = np.abs(coeffs) ** 2   # wavelet power spectrum

    # ── 6.2 Plot scalogram ───────────────────────────────────
    fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                              gridspec_kw={'height_ratios': [1, 2.5]})

    # Top: time series
    axes[0].fill_between(years, y_norm, color=COLORS['amber'], alpha=0.4)
    axes[0].plot(years, y_norm, color=COLORS['amber'], lw=1.0)
    axes[0].set_ylabel('Std. Sunspots')
    axes[0].set_title('Sunspot Activity and Wavelet Power Spectrum (Scalogram)')
    axes[0].set_xlim(years[0], years[-1])

    # Bottom: scalogram
    periods_wt = 1.0 / freqs_cwt
    im = axes[1].contourf(years, np.log2(periods_wt), np.log2(power + 1e-8),
                           levels=20, cmap='YlOrRd')
    axes[1].axhline(np.log2(11), color='white', lw=1.5, ls='--',
                    label='11-year period')
    axes[1].set_xlabel('Year')
    axes[1].set_ylabel('Period (years, log2 scale)')

    # Y-axis ticks in years
    yticks = [2, 4, 8, 16, 32]
    axes[1].set_yticks(np.log2(yticks))
    axes[1].set_yticklabels([str(y) for y in yticks])
    axes[1].legend(loc='upper right')

    plt.colorbar(im, ax=axes[1], label='log2(Power)')
    plt.tight_layout()
    plt.show()
    print('Scalogram shows the ~11-year solar cycle persists across the entire record.')
    print('Warmer colors (red/orange) = higher power at that period and time.')
else:
    print('PyWavelets not installed. Run: pip install PyWavelets')

In [ ]:
# ── 6.3 Wavelet analysis of GDP business cycle ───────────────
if HAS_PYWT:
    y_gdp      = log_gdp.values.astype(float)
    y_gdp_norm = (y_gdp - y_gdp.mean()) / y_gdp.std()

    # Periods: 4 to 48 quarters
    freqs_gdp  = np.geomspace(1/48, 1/4, 50)
    scales_gdp = pywt.frequency2scale(wavelet, freqs_gdp)

    coeffs_gdp, freqs_cwt_gdp = pywt.cwt(y_gdp_norm, scales_gdp, wavelet,
                                           sampling_period=1.0)
    power_gdp  = np.abs(coeffs_gdp) ** 2
    periods_gdp = 1.0 / freqs_cwt_gdp

    fig, axes = plt.subplots(2, 1, figsize=(14, 9),
                              gridspec_kw={'height_ratios': [1, 2.5]})

    axes[0].plot(log_gdp.index, y_gdp, color=COLORS['blue'], lw=1.5)
    axes[0].set_ylabel('Log GDP')
    axes[0].set_title('Simulated GDP and Wavelet Power Spectrum')

    im = axes[1].contourf(log_gdp.index, np.log2(periods_gdp),
                           np.log2(power_gdp + 1e-8),
                           levels=20, cmap='YlOrRd')
    # Business cycle band markers
    axes[1].axhline(np.log2(32), color='white', lw=1.5, ls='--', label='32 quarters')
    axes[1].axhline(np.log2(6),  color='white', lw=1.5, ls=':',  label='6 quarters')
    axes[1].set_xlabel('Date')
    axes[1].set_ylabel('Period (quarters, log2 scale)')

    yticks_q = [4, 6, 8, 12, 16, 24, 32, 48]
    axes[1].set_yticks(np.log2(yticks_q))
    axes[1].set_yticklabels([str(q) for q in yticks_q])
    axes[1].legend(loc='upper right')

    plt.colorbar(im, ax=axes[1], label='log2(Power)')
    plt.tight_layout()
    plt.show()
    print('Scalogram reveals the dominant 20-quarter business cycle across the simulation.')
    print('White dashed lines mark the NBER business cycle band: 6-32 quarters.')
else:
    print('PyWavelets not installed. Run: pip install PyWavelets')

---
## Section 7: Summary and Key Takeaways

### Core Concepts Covered

| Topic | Key Idea | Python Tool |
|-------|----------|-------------|
| **DFT** | Decomposes series into sinusoids | `scipy.fft.fft` |
| **Periodogram** | Squared DFT — inconsistent spectral estimator | `scipy.signal.periodogram` |
| **Welch's method** | Averaged overlapping segment periodograms | `scipy.signal.welch` |
| **Multitaper** | Orthogonal DPSS tapers, minimum leakage | `scipy.signal.windows.dpss` |
| **Blackman-Tukey** | Windowed autocovariance -> FFT | Manual implementation |
| **Coherence** | Cross-spectral $R^2$ — frequency-band correlation | `scipy.signal.coherence` |
| **HP filter** | High-pass filter, $\lambda=1600$ (quarterly) | `statsmodels hpfilter` |
| **Baxter-King** | Symmetric MA, approximate ideal band-pass | `statsmodels bkfilter` |
| **Christiano-Fitzgerald** | Asymmetric MA, uses full sample | `statsmodels cffilter` |
| **Wavelet CWT** | Time-frequency representation | `pywt.cwt` |
| **Scalogram** | $|W(a,b)|^2$ — wavelet power vs time and period | `pywt.cwt` + `contourf` |

### When to Use Each Method

- **Periodogram:** Quick initial look at frequency content
- **Welch/multitaper:** Need reliable PSD estimate for inference
- **HP filter:** Simple trend removal (but leaks some low-frequency noise)
- **BK/CF filters:** Business cycle extraction with frequency-band precision
- **Wavelet:** Non-stationary series where cycle amplitude/period varies over time
- **Coherence:** Measure frequency-specific co-movement between two series

### Next Steps
- **Seminar:** Practice exercises on sunspot cycles, GDP business cycles, and financial volatility
- **Advanced topics:** Long-memory estimation via spectral regression, wavelet coherence, spectral change-point detection